# Lab 1.4 &mdash; The Coordination Tax

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Build the same capability twice: one agent, and three specialists
- Instrument both &mdash; calls, tokens, latency &mdash; before arguing about either
- Score them on one eval set and let the result be whatever it is

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 1 labs work one case: a small tech-support ticket queue.
> What you build in each lab is picked up by the next one.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# A small tech-support ticket queue. Ordinary rules on purpose: the only new thing in these
# five labs is LangChain. Nothing here is real data and nothing leaves this notebook.

TICKETS = {
    "TCK-4001": {"customer": "Priya Nair",   "product": "VPN Client", "version": "4.2",
                 "severity": "high",   "error_code": "VPN-513",
                 "text": "Cannot connect since the upgrade. Error VPN-513."},
    "TCK-4002": {"customer": "Rahul Menon",  "product": "Reports",    "version": "3.9.1",
                 "severity": "low",    "error_code": "APP-002",
                 "text": "Monthly export finishes but the PDF is blank."},
    "TCK-4003": {"customer": "Anita Sharma", "product": "Reports",    "version": "3.9.1",
                 "severity": "medium", "error_code": None,
                 "text": "It is just slow today. Nothing else to add."},
    "TCK-4004": {"customer": "Vikram Rao",   "product": "VPN Client", "version": "4.2",
                 "severity": "high",   "error_code": "SEC-900",
                 "text": "Got a login alert from a country I have never visited."},
    "TCK-4005": {"customer": "Priya Nair",   "product": "Reports",    "version": "3.9.1",
                 "severity": "low",    "error_code": "APP-002",
                 "text": "Same blank PDF as my colleague reported."},
}

# The runbook: what support is allowed to do about each error code.
RUNBOOK = {
    "VPN-513": "Certificate pinning changed in 4.2. Have the user clear the local trust store "
               "and re-enrol. Five minutes, no data loss. Support may do this without approval.",
    "APP-002": "Known defect in 3.9.1, fixed in 3.9.2. Advise the upgrade. Do not issue a refund "
               "for this and do not raise a new defect -- link the existing one.",
    "SEC-900": "Possible credential compromise. Escalate to the security desk immediately. "
               "Support must not resolve, close or advise the customer directly.",
}

# Which error codes may an agent resolve on its own, and which need a human?
MUST_ESCALATE = {"SEC-900"}

print(f"{len(TICKETS)} tickets, {len(RUNBOOK)} runbook entries loaded")

## Concept

&ldquo;Split it into specialists&rdquo; sounds obviously right. It is a design decision with a
price, and the price is rarely the one people quote.

You are going to build both, measure both, and then decide. The rule for this lab: **the team is
allowed to lose.** That outcome is the lesson, not a lab failure.

## Section 1 &mdash; Two architectures over the same tools

Same model, same three tools, same questions. The only difference is how the work is divided.

Deciding **which tools each specialist gets** is the entire design of a split &mdash; and giving
one of them too little is how a handoff starts losing information.

In [ ]:
import json, time
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def lookup_ticket(ref: str) -> str:
    """Return the support ticket for one reference such as 'TCK-4001': customer, product,
    version, severity and error code."""
    t = TICKETS.get(ref)
    return json.dumps({"ref": ref, **t}) if t else f"no ticket {ref!r}"

@tool
def runbook_for(error_code: str) -> str:
    """Return what support is allowed to do about one error code, e.g. 'VPN-513', including
    whether it must be escalated."""
    return RUNBOOK.get(error_code, f"no runbook entry for {error_code!r}")

@tool
def find_similar(error_code: str) -> str:
    """Return the references of other tickets reporting the same error code."""
    return json.dumps([r for r, t in TICKETS.items() if t["error_code"] == error_code])

ALL_TOOLS = {"lookup_ticket": lookup_ticket, "runbook_for": runbook_for, "find_similar": find_similar}


def specialist_tools() -> dict:
    """Which tools does each specialist need to do its own job -- and only its own job?"""
    return {
        "triage":   BLANK,   # TODO: reads the ticket and reports what it is. Which tool(s)?
        "diagnose": BLANK,   # TODO: given an error code, works out what is allowed. Which tool(s)?
        "resolve":  BLANK,   # TODO: writes the answer. Does it need a tool at all?
    }

In [ ]:
ROLES = {
    "triage":   "Read the ticket and state the error code and severity. Two lines maximum.",
    "diagnose": "Given an error code, state what support is allowed to do and whether it escalates.",
    "resolve":  "Write the final answer for the customer from what you were told. Three lines maximum.",
}

def make_agent(names, instructions):
    return create_agent(model=get_llm(), tools=[ALL_TOOLS[n] for n in names],
                        system_prompt=instructions)

def usage(out):
    """Calls, tokens and the final text out of an agent result. Given -- you are measuring."""
    msgs = out["messages"]
    calls = sum(len(getattr(m, "tool_calls", []) or []) for m in msgs)
    toks = sum((getattr(m, "usage_metadata", None) or {}).get("total_tokens", 0) for m in msgs)
    return calls, toks, msgs[-1].content


# --- Self-check: Section 1   (the split, and that it names real tools -- no model needed)
check("every specialist names tools that exist",
      lambda: all(n in ALL_TOOLS for names in specialist_tools().values() for n in names))
check("triage can read a ticket and diagnose can read the runbook",
      lambda: ("lookup_ticket" in specialist_tools()["triage"]
               and "runbook_for" in specialist_tools()["diagnose"]))
check("no specialist got every tool",
      lambda: all(len(v) < len(ALL_TOOLS) for v in specialist_tools().values()),
      "if one of them has all three you have built one agent with extra steps")
score()

## Section 2 &mdash; One eval set, both arms

Five questions with an answer you can check by substring. Crude on purpose: a number you actually
compute beats a number you assert.

Then the part that matters &mdash; deciding **what the comparison is for**. Write the rule before
you see the result.

In [ ]:
EVAL = [
    ("What should we do about TCK-4001?", "trust store"),
    ("What should we do about TCK-4002?", "3.9.2"),
    ("What should we do about TCK-4004?", "security"),
    ("What should we do about TCK-4005?", "3.9.2"),
    ("Can support close TCK-4004?",       "security"),
]

def verdict(single: dict, team: dict) -> str:
    """single and team each look like {"passed": int, "tokens": int, "seconds": float}."""
    cheaper_wins  = "team" if team["tokens"] < single["tokens"] else "single"
    quality_first = ("team" if team["passed"] > single["passed"] else
                     "single" if single["passed"] > team["passed"] else
                     ("team" if team["tokens"] < single["tokens"] else "single"))

    return BLANK        # TODO: which rule would you defend in a design review?

In [ ]:
# --- Self-check: Section 2   (the decision rule, on numbers you invent -- no model)
def r(passed, tokens, seconds=1.0):
    return {"passed": passed, "tokens": tokens, "seconds": seconds}

check("a better answer wins even when it costs more",
      lambda: verdict(r(3, 5000), r(5, 9000)) == "team")
check("a cheaper wrong answer does not win",
      lambda: verdict(r(5, 9000), r(2, 3000)) == "single",
      "this is the rule that stops 'the team is cheaper' ending a design review")
check("cost settles a tie",
      lambda: verdict(r(5, 9000), r(5, 4000)) == "team")
score()

## Run it for real &mdash; both arms, one eval set

In [ ]:
def run_single(q):
    a = make_agent(list(ALL_TOOLS), "You are a tech support analyst. Answer the question fully.")
    t0 = time.time()
    calls, toks, text = usage(a.invoke({"messages": [("user", q)]}))
    return calls, toks, time.time() - t0, text

def run_team(q):
    """Three agents, each handing the next one prose -- which is the point."""
    ag = {role: make_agent(specialist_tools()[role], ROLES[role]) for role in ROLES}
    t0, calls, toks = time.time(), 0, 0
    handoff = q
    for role in ["triage", "diagnose", "resolve"]:
        c, k, handoff = usage(ag[role].invoke({"messages": [("user", handoff)]}))
        calls, toks = calls + c, toks + k
    return calls, toks, time.time() - t0, handoff

def arm(runner):
    passed = calls = toks = 0; secs = 0.0
    for q, want in EVAL:
        c, k, s, text = runner(q)
        ok = want.lower() in (text or "").lower()
        passed += ok; calls += c; toks += k; secs += s
        print(f"   {'ok  ' if ok else 'MISS'} {q[:34]:36} want={want:12} {c} calls, {k} tokens")
    return {"passed": passed, "calls": calls, "tokens": toks, "seconds": secs}


if llm_ready():
    def bakeoff():
        print("--- one agent, three tools ---");    single = arm(run_single)
        print("--- three specialists ---");         team   = arm(run_team)
        print(f"\n{'':10}{'passed':>8}{'calls':>8}{'tokens':>9}{'seconds':>9}")
        for name, m in [("single", single), ("team", team)]:
            print(f"{name:10}{m['passed']:>6}/5{m['calls']:>8}{m['tokens']:>9}{m['seconds']:>9.1f}")
        print("\nverdict:", verdict(single, team))
    guard(bakeoff)

### Read it

Whatever your numbers were, read them against the claim everybody repeats &mdash; that multi-agent
costs **3&ndash;10&times;** the tokens.

On this sandbox it does not. Measured over this eval set, tokens come out roughly **level**; what
actually multiplies is **calls** (about 2&times;) and **latency** (about 1.5&times;), and what
falls is the **pass rate**. The coordination tax is real and it is not mainly a token bill.

**Where the quality goes.** Each specialist hands the next one prose. The error code survives that
&mdash; check it &mdash; but the *context* does not: `resolve` never saw the ticket, only a summary
of a summary. That is fragmentation, not data loss, and it is why Module 5 hands structured state
between agents rather than sentences.

**So the honest answer here is &ldquo;one agent&rdquo;**, and being able to say that with a number
behind it is the point of Module 1. Lab 1.5 turns it into a rule you can apply before building.

In [ ]:
score()

## Your turn

1. Give `resolve` the `lookup_ticket` tool so it can re-read the ticket instead of trusting the
   handoff. Does the pass rate recover? What did it cost?
2. Hand a dict between the specialists instead of prose &mdash; ticket, error code, runbook text.
   You have just invented Module 5's shared state, and you can measure what it bought.